# Step 1: Collect the documents (about 45 minutes)

**What you are building this week:** a RAG (retrieval-augmented generation) assistant that answers questions about Databricks using its public documentation, and shows sources for every answer.

**Where Step 1 fits.** Every RAG system has two halves:

| Half | When it runs | Steps |
|---|---|---|
| **Ingestion (offline)** | Once, then on updates | Collect documents → clean → split into chunks → embed → store in an index |
| **Query (online)** | Every question | Embed the question → retrieve chunks → build a prompt → generate an answer with citations |

Step 1 is the first box: **collect and clean the documents.** It sounds boring, but it decides the ceiling of the whole system. If the text is noisy or missing, no embedding model or prompt can fix it. In production, most RAG failures trace back to ingestion.

**This step costs nothing.** It makes no OpenAI calls.

## 1.1 Setup: packages and where to save files

**Concepts**
- `requests` downloads a web page (it sends an HTTP GET request and gets back the HTML).
- `beautifulsoup4` (BeautifulSoup) parses HTML so you can find elements such as the title, the main article and the links.
- **Colab forgets everything when the session ends.** To keep the collected documents for Steps 2-10, save them to Google Drive. Set `USE_DRIVE = True` (recommended) and approve the Drive pop-up. With `False`, files live only in this session.

In [ ]:
%pip install -q requests beautifulsoup4
import os, re, json, time, datetime
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urldefrag, urlparse
from urllib import robotparser

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/docs-rag"
else:
    BASE = "/content/docs-rag"
DATA = f"{BASE}/data"
os.makedirs(DATA, exist_ok=True)
print("Files will be saved in:", DATA)

Mounted at /content/drive
Files will be saved in: /content/drive/MyDrive/docs-rag/data


**What you should see:** `Files will be saved in: /content/drive/MyDrive/docs-rag/data`. We will reuse this folder in every later step.

## 1.2 Choose the sources and set the rules

**Concepts**
- **Seed URLs** are the starting pages. We pick three sections of the Databricks docs that relate to governance and AI, which match the job description.
- **Scope:** we only follow links that stay *inside* each seed's section (the URL must start with the seed). Without this, a crawler wanders across the whole site.
- **Limit:** at most 15 pages per seed. For a first RAG system, a small, relevant corpus you can evaluate beats a huge one you can't.
- **Politeness:**
  - a pause between requests (`DELAY`) so you don't overload the server;
  - a clear `User-Agent` header saying who you are;
  - respecting `robots.txt`, the file where a site lists what automated tools may fetch.


In [ ]:
SEEDS = [
    "https://docs.databricks.com/aws/en/data-governance/unity-catalog/",
    "https://docs.databricks.com/aws/en/ai-gateway/",
    "https://docs.databricks.com/aws/en/catalog-explorer/",
]
MAX_PER_SEED = 15
DELAY = 0.5   # seconds between requests
HEADERS = {"User-Agent": "Mozilla/5.0 (personal learning project; docs-rag)"}

rp = robotparser.RobotFileParser("https://docs.databricks.com/robots.txt")
try:
    rp.read()
    for s in SEEDS:
        print("allowed" if rp.can_fetch(HEADERS["User-Agent"], s) else "BLOCKED", s)
except Exception as e:
    print("Could not read robots.txt:", e)

allowed https://docs.databricks.com/aws/en/data-governance/unity-catalog/
allowed https://docs.databricks.com/aws/en/ai-gateway/
allowed https://docs.databricks.com/aws/en/catalog-explorer/


## 1.3 Fetch one page and look at it

**Concepts**
- An HTTP request returns a **status code**: 200 means OK, 404 means not found, 429 means "too many requests" (slow down), 5xx means a server error.
- `raise_for_status()` turns a bad status into a Python error, so failures are visible instead of silent.
- `timeout=30` stops the code from hanging forever on a slow page.
- We retry a couple of times with a growing wait (**backoff**), because networks fail occasionally.

Always inspect one example by hand before automating. It is the fastest way to catch problems.

In [ ]:
def get_html(url, tries=3):
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            r.raise_for_status()
            if "text/html" not in r.headers.get("Content-Type", ""):
                return None                       # skip images, PDFs, etc.
            return BeautifulSoup(r.text, "html.parser")
        except requests.RequestException as e:
            if attempt == tries:
                print("  failed:", url, "|", e)
                return None
            time.sleep(2 * attempt)               # backoff: wait 2s, then 4s

soup = get_html(SEEDS[0])
print("Title:", soup.title.get_text(strip=True))
print("Raw HTML length (characters):", len(str(soup)))
print("All visible text length:", len(soup.get_text()))

Title: What is Unity Catalog? | Databricks on AWS
Raw HTML length (characters): 25763
All visible text length: 4585


**What you should see:** a title like "What is Unity Catalog? | Databricks on AWS". The raw HTML is much longer than the text: most of a web page is markup, scripts and styling.

## 1.4 Clean the text

**Concepts**
- A docs page contains the article plus **boilerplate**: menus, headers, footers, cookie banners, "On this page" lists. Boilerplate repeats on every page.
- **Why remove it:** if you keep it, every page shares the same menu text. Those repeated words end up inside chunks and embeddings, so a question can retrieve a chunk full of menu items instead of the real answer. This is called noise.
- **How:** documentation sites usually put the real content inside an `<article>` or `<main>` element. We take that element and delete navigation, scripts and buttons inside it.
- **Keep sentences whole.** Links sit inside sentences (`See <a>Securable objects</a> for details`). If we split text at every HTML element, sentences break into fragments. Instead we mark the end of each block element (paragraph, list item, heading), join everything else with spaces, and then split only at those marks: one clean line per paragraph.

In [ ]:
BLOCKS = ["p", "li", "h1", "h2", "h3", "h4", "h5", "h6", "pre", "tr", "table", "div", "br"]
MARK = "@@BR@@"

def clean_text(soup):
    node = soup.find("article") or soup.find("main") or soup.body
    for tag in node(["script", "style", "nav", "footer", "header", "button", "svg", "form"]):
        tag.decompose()                                   # delete the element and its contents
    for el in node.find_all(BLOCKS):
        el.append(MARK)                                   # mark where each block ends
    text = re.sub(r"\s+", " ", node.get_text())          # join inline text (links stay in sentences)
    lines = [ln.strip() for ln in text.split(MARK)]       # one line per block
    return "\n".join(ln for ln in lines if ln)

def page_title(soup):
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else soup.title.get_text(strip=True)

soup = get_html(SEEDS[0])
txt = clean_text(soup)
print("Title:", page_title(soup))
print("Clean text length:", len(txt), "characters,", len(txt.split()), "words")
print("-" * 60)
print(txt[:1200])

Title: What is Unity Catalog? | Databricks on AWS
Clean text length: 3861 characters, 579 words
------------------------------------------------------------
Last updated on Sep 11, 2026
Unity Catalog is the unified governance layer for data and AI built into Databricks. When enabled for a workspace, Unity Catalog operates beneath every data and AI interaction in your workspaces automatically: enforcing access control when you query a table or call a model, tracking lineage as data and AI assets are used, logging activity for auditing, and more. You work with the objects Unity Catalog governs through Catalog Explorer, SQL, the Databricks CLI, and REST APIs.
Unity Catalog is automatically enabled for all Databricks workspaces created after November 8, 2023.
To verify that Unity Catalog is enabled and set up properly for your workspace, see Unity Catalog setup guide.
If your workspace was created before November 8, 2023, see Upgrade a Databricks workspace to Unity Catalog.
Unity Catalog i

## 1.5 Discover links within a section

**Concepts**
- Links in HTML are `<a href="...">` elements.
- **Relative vs absolute URLs:** `href="securable-objects"` is relative; `urljoin` turns it into the full address.
- **Fragments:** `page#section` is the same page as `page`, so we strip the `#...` part with `urldefrag` to avoid fetching duplicates.
- We keep only links that start with the seed URL (scope rule from 1.2).

In [ ]:
def section_links(seed, soup):
    found = []
    for a in soup.find_all("a", href=True):
        url = urldefrag(urljoin(seed, a["href"]))[0]
        if url.startswith(seed) and url not in found:
            found.append(url)
    return found

links = section_links(SEEDS[0], get_html(SEEDS[0]))
print(len(links), "links inside the Unity Catalog section")
for u in links[:10]:
    print(" ", u)

12 links inside the Unity Catalog section
  https://docs.databricks.com/aws/en/data-governance/unity-catalog/
  https://docs.databricks.com/aws/en/data-governance/unity-catalog/setup-uc
  https://docs.databricks.com/aws/en/data-governance/unity-catalog/upgrade/
  https://docs.databricks.com/aws/en/data-governance/unity-catalog/securable-objects
  https://docs.databricks.com/aws/en/data-governance/unity-catalog/managed-versus-external
  https://docs.databricks.com/aws/en/data-governance/unity-catalog/requirements
  https://docs.databricks.com/aws/en/data-governance/unity-catalog/access-control/
  https://docs.databricks.com/aws/en/data-governance/unity-catalog/data-lineage
  https://docs.databricks.com/aws/en/data-governance/unity-catalog/data-classification
  https://docs.databricks.com/aws/en/data-governance/unity-catalog/data-quality-monitoring/


## 1.6 Crawl all three sections

**Concepts**
- This loop repeats 1.3-1.5 for each seed: fetch the seed, find its section links, then fetch up to 15 pages.
- `docs` is a dictionary keyed by URL, so the same page is never stored twice.
- For each page we store **metadata**: URL, title, which section it came from, word count and the crawl date.
- **Why metadata matters:** the URL and title become the **citations** in answers; the section lets you **filter** searches later; the crawl date tells you how **fresh** the index is. In a company setting you would also store an access group so users only retrieve documents they're allowed to see.

This takes about 1-2 minutes because of the pause between requests.

In [ ]:
crawl_date = datetime.date.today().isoformat()
docs = {}
for seed in SEEDS:
    seed_soup = get_html(seed)
    if seed_soup is None:
        continue
    urls = [seed] + [u for u in section_links(seed, seed_soup) if u != seed]
    section = urlparse(seed).path.strip("/").split("/")[-1]
    added = 0
    for url in urls:
        if added >= MAX_PER_SEED:
            break
        if url in docs:
            continue
        soup = seed_soup if url == seed else get_html(url)
        if soup is None:
            continue
        text = clean_text(soup)
        if len(text.split()) < 50:          # skip near-empty pages
            continue
        docs[url] = {"url": url, "title": page_title(soup), "section": section,
                     "words": len(text.split()), "crawled": crawl_date, "text": text}
        added += 1
        time.sleep(DELAY)
    print(f"{section}: {added} pages")
print("Total pages:", len(docs))

unity-catalog: 12 pages
ai-gateway: 15 pages
catalog-explorer: 1 pages
Total pages: 28


## 1.7 Check the quality of what you collected

**Concepts**
- Never trust a pipeline you haven't inspected. Look at sizes, the smallest pages and possible duplicates.
- **Very short pages** are often index pages with only links. They add little and can be dropped.
- **Duplicates:** two URLs can serve the same text. We detect exact duplicates with a **hash** (a fingerprint of the text). Near-duplicates need fuzzier methods such as MinHash; mention that in interviews.

In [ ]:
import hashlib, statistics
words = [d["words"] for d in docs.values()]
print(f"Pages: {len(docs)} | total words: {sum(words):,} | median words/page: {statistics.median(words):.0f}")
print("\nSmallest pages:")
for d in sorted(docs.values(), key=lambda d: d["words"])[:5]:
    print(f"  {d['words']:>5} words  {d['title']}")
print("\nLargest pages:")
for d in sorted(docs.values(), key=lambda d: -d["words"])[:3]:
    print(f"  {d['words']:>5} words  {d['title']}")

hashes = {}
for u, d in docs.items():
    h = hashlib.md5(d["text"].encode()).hexdigest()
    hashes.setdefault(h, []).append(u)
dups = [v for v in hashes.values() if len(v) > 1]
print("\nExact duplicate groups:", len(dups))
for g in dups:
    print("  ", g)

Pages: 28 | total words: 40,545 | median words/page: 948

Smallest pages:
    244 words  What is Catalog Explorer? | Databricks on AWS
    277 words  Use open-source models | Databricks on AWS
    400 words  Access control in Unity Catalog | Databricks on AWS
    424 words  Get started with Unity Catalog | Databricks on AWS
    579 words  What is Unity Catalog? | Databricks on AWS

Largest pages:
   5805 words  Unity Catalog securable objects reference | Databricks on AWS
   3542 words  Unity Catalog best practices | Databricks on AWS
   2978 words  Track model usage | Databricks on AWS

Exact duplicate groups: 0


## 1.8 Save the corpus

**Concepts**
- JSON is a simple, readable format, fine for a few dozen pages. At company scale you would write to a table, for example a **Delta table** on Databricks, so the corpus is versioned and queryable.
- **Do not commit the downloaded text to GitHub.** Commit this notebook instead, so anyone can rebuild the data. That is the reproducible way, and it avoids republishing someone else's content.

In [ ]:
path = f"{DATA}/docs.json"
with open(path, "w") as f:
    json.dump(docs, f)
print("Saved", len(docs), "pages to", path, f"({os.path.getsize(path)/1024:.0f} KB)")

# reload to confirm the file is good
check = json.load(open(path))
first = next(iter(check.values()))
print("Reloaded OK. Example:", first["title"], "|", first["section"], "|", first["words"], "words")

Saved 28 pages to /content/drive/MyDrive/docs-rag/data/docs.json (290 KB)
Reloaded OK. Example: What is Unity Catalog? | Databricks on AWS | unity-catalog | 579 words
